# Tarea 3 — Autoencoders VAE y U-Net · ITCR

## Cómo ejecutar
1. Configure **`.env.example`** en la raíz (API key, proyecto, entity).
2. **Kernel → Restart**
3. **Run All** (celdas en orden)

| Celda | Contenido |
|-------|-----------|
| 1–2 | Dependencias e imports |
| 3–4 | W&B + Hydra (`conf/`) |
| 5–6 | Dataset MVTec |
| 7 | Modelos, callback y funciones de entrenamiento |
| 8 | **Entrena 8 experimentos** |
| 9–10 | Gráficas y resultados W&B |
| 11 | Análisis |

**Configuración Hydra:** solo `conf/` y `.env` quedan fuera.
**Gráficas:** solo desde W&B (prohibido matplotlib/plotly local).


## 1 — Dependencias

In [1]:
import subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "conf").exists() and (PROJECT_ROOT.parent / "conf").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

try:
    import hydra, pytorch_lightning
    print("OK", sys.executable)
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")])
    print("Reinicie el kernel.")

W0524 05:38:04.437000 31064 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


OK c:\Users\boyfa\miniconda3\python.exe


## 2 — Imports, `.env` y GPU

In [2]:
import json
import os
import sys
from pathlib import Path
from typing import Any, Dict, Iterator, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import pytorch_lightning as pl
import wandb
from hydra import compose, initialize_config_dir
from omegaconf import DictConfig, OmegaConf
from IPython.display import HTML, Image, display
from pytorch_lightning import Callback, LightningModule, LightningDataModule, Trainer
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.utilities import rank_zero_only
from PIL import Image as PILImage
from pytorch_msssim import ssim
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms

CONF_DIR = str((PROJECT_ROOT / "conf").resolve())
DATASET_ROOT = PROJECT_ROOT / "dataset"

"""Configuración W&B: lee .env.example y .env (este último tiene prioridad)."""

import os
from pathlib import Path
from typing import List, Optional, Tuple


def _project_root(project_root: Optional[Path] = None) -> Path:
    root = project_root or Path.cwd()
    if not (root / "conf").exists() and (root.parent / "conf").exists():
        root = root.parent
    return root


def load_dotenv_if_present(project_root: Optional[Path] = None) -> List[str]:
    """
    Carga variables WANDB_* desde .env.example y luego .env (override).
    Devuelve lista de archivos leídos.
    """
    root = _project_root(project_root)
    loaded: List[str] = []

    for name in (".env.example", ".env"):
        env_file = root / name
        if not env_file.exists():
            continue
        loaded.append(name)
        for line in env_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, value = line.partition("=")
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            if not key.startswith("WANDB_"):
                continue
            if "paste_your" in value.lower():
                continue
            # Permite WANDB_ENTITY= vacío → cuenta por defecto
            os.environ[key] = value

    return loaded


def print_env_status(project_root: Optional[Path] = None) -> None:
    files = load_dotenv_if_present(project_root)
    key = os.environ.get("WANDB_API_KEY", "")
    masked = (key[:8] + "..." + key[-4:]) if len(key) > 12 else ("(no configurada)" if not key else "***")
    print("Archivos cargados:", files or ["ninguno — cree .env.example"])
    print("WANDB_API_KEY:", masked)
    print("WANDB_ENTITY:", os.environ.get("WANDB_ENTITY") or "(vacío → cuenta por defecto)")
    print("WANDB_PROJECT:", os.environ.get("WANDB_PROJECT", "(no definido)"))


def ensure_wandb_login() -> None:
    import wandb

    api_key = os.environ.get("WANDB_API_KEY")
    if not api_key:
        raise RuntimeError(
            "WANDB_API_KEY no encontrada. Configure .env.example en la raíz del proyecto."
        )
    wandb.login(key=api_key, relogin=True)


def _normalize_entity(entity: Optional[str]) -> Optional[str]:
    if entity is None:
        return None
    s = str(entity).strip()
    if not s or s.lower() in ("null", "none", "~"):
        return None
    return s


_RESOLVED: Optional[Tuple[str, Optional[str]]] = None


def resolve_wandb_settings(
    project: str,
    entity: Optional[str] = None,
    project_root: Optional[Path] = None,
    force: bool = False,
) -> Tuple[str, Optional[str]]:
    global _RESOLVED
    if _RESOLVED is not None and not force:
        return _RESOLVED

    import wandb

    load_dotenv_if_present(project_root)
    ensure_wandb_login()

    project = os.environ.get("WANDB_PROJECT") or project
    entity = _normalize_entity(os.environ.get("WANDB_ENTITY") or entity)

    api = wandb.Api()
    default_entity = getattr(api, "default_entity", None)

    def _try_init(ent: Optional[str]):
        run = wandb.init(project=project, entity=ent, job_type="preflight", reinit=True)
        used = run.entity if run and hasattr(run, "entity") else ent
        wandb.finish()
        return used

    if entity:
        try:
            used = _try_init(entity)
            print(f"W&B OK — entity={used}, project={project}")
            _RESOLVED = (project, used)
            return _RESOLVED
        except Exception as exc:
            msg = str(exc).lower()
            if "permission" in msg or "denied" in msg or "not found" in msg:
                print(f"Sin permiso en '{entity}'. Usando cuenta por defecto: {default_entity}")
            else:
                raise

    used = _try_init(None)
    final_entity = used or default_entity
    print(f"W&B OK — entity={final_entity}, project={project}")
    _RESOLVED = (project, final_entity)
    return _RESOLVED


load_dotenv_if_present(PROJECT_ROOT)
print_env_status(PROJECT_ROOT)

print("\nProyecto:", PROJECT_ROOT)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Archivos cargados: ['.env']
WANDB_API_KEY: wandb_v1...JQl8
WANDB_ENTITY: fabriciomena11-tec
WANDB_PROJECT: Tarea_3

Proyecto: C:\Users\boyfa\OneDrive\Documentos\AI\Tarea_3\Repo_tarea_3\AI_Tarea3
CUDA: True
GPU: NVIDIA GeForce RTX 5060


## 3 — Configuración Hydra (`conf/`)

In [3]:
def load_cfg(overrides=None):
    overrides = list(overrides or [])
    if not any(o.startswith("data.root=") for o in overrides):
        overrides.append("data.root=" + DATASET_ROOT.as_posix())
    with initialize_config_dir(version_base=None, config_dir=CONF_DIR):
        return compose(config_name="config", overrides=overrides)

cfg = load_cfg()
print(OmegaConf.to_yaml(cfg))

model:
  name: vae
  latent_dim: 128
  learning_rate: 0.0001
  kl_weight: 0.001
trainer:
  max_epochs: 30
  accelerator: gpu
  devices: 1
  precision: 16-mixed
  gradient_clip_val: 1.0
  log_every_n_steps: 10
  check_val_every_n_epoch: 1
  enable_progress_bar: true
logger:
  project: ${oc.env:WANDB_PROJECT,Tarea_3}
  entity: ${oc.env:WANDB_ENTITY,fabriciomena11-tec}
  tags:
  - mvtec
  - tarea-3
  - ${model.name}
  - ${loss.name}
  log_model: false
  save_dir: wandb_logs
data:
  root: C:/Users/boyfa/OneDrive/Documentos/AI/Tarea_3/Repo_tarea_3/AI_Tarea3/dataset
  classes:
  - cable
  - capsule
  - screw
  - transistor
  image_size: 128
  num_channels: 3
  batch_size: 16
  num_workers: 0
  use_predefined_splits: true
  val_source: test_good
  val_folder_names:
  - validation
  - val
  val_split: 0.1
loss:
  name: l1
  type: l1
experiments:
  models:
  - vae
  - unet
  losses:
  - l1
  - l2
  - ssim
  - ssim_l1
seed: 42
experiment_name: ${model.name}_${loss.name}



## 4 — Dataset MVTec (código + verificación)

In [4]:
from pathlib import Path
from typing import List, Optional

import torch
from PIL import Image as PILImage
from pytorch_lightning import LightningDataModule
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms


def _list_images(folder: Path) -> List[Path]:
    if not folder.exists():
        return []
    exts = {".png", ".jpg", ".jpeg", ".bmp"}
    return sorted(p for p in folder.rglob("*") if p.suffix.lower() in exts)


def _resolve_val_folder(cls_root: Path, val_folder_names: List[str]) -> Optional[Path]:
    for name in val_folder_names:
        folder = cls_root / name
        if folder.is_dir() and _list_images(folder):
            return folder
    return None


def _has_test_good(root: Path, classes: List[str]) -> bool:
    for cls in classes:
        good_dir = root / cls / "test" / "good"
        if good_dir.is_dir() and _list_images(good_dir):
            return True
    return False


class MVTecImageDataset(Dataset):
    """Carga imágenes MVTec AD: train/good, validation o test/good, test completo."""

    def __init__(
        self,
        root: str,
        classes: List[str],
        image_size: int = 128,
        split: str = "train",
        val_folder_names: Optional[List[str]] = None,
        val_source: str = "test_good",
        defect_only: bool = False,
        good_only: bool = False,
    ):
        self.root = Path(root)
        self.image_size = image_size
        self.paths: List[Path] = []
        self.labels: List[str] = []
        val_folder_names = val_folder_names or ["validation", "val"]

        for cls in classes:
            cls_root = self.root / cls
            if split == "train":
                folders = [cls_root / "train"]
            elif split == "validation":
                val_dir = _resolve_val_folder(cls_root, val_folder_names)
                if val_dir:
                    folders = [val_dir]
                elif val_source == "test_good":
                    folders = [cls_root / "test" / "good"]
                else:
                    folders = []
            elif split == "test":
                if defect_only:
                    test_root = cls_root / "test"
                    if test_root.exists():
                        for sub in sorted(test_root.iterdir()):
                            if sub.is_dir() and sub.name != "good":
                                self._add_folder(sub, f"{cls}/{sub.name}")
                    continue
                elif good_only:
                    folders = [cls_root / "test" / "good"]
                else:
                    test_root = cls_root / "test"
                    if test_root.exists():
                        for sub in sorted(test_root.iterdir()):
                            if sub.is_dir():
                                label = "good" if sub.name == "good" else f"{cls}/{sub.name}"
                                self._add_folder(sub, label)
                    continue
            else:
                raise ValueError(f"Unknown split: {split}")

            for folder in folders:
                if folder and folder.exists():
                    self._add_folder(folder, f"{cls}/good")

        self.transform = transforms.Compose(
            [
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
            ]
        )

    def _add_folder(self, folder: Path, label: str) -> None:
        for path in _list_images(folder):
            self.paths.append(path)
            self.labels.append(label)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        img = PILImage.open(self.paths[idx]).convert("RGB")
        return self.transform(img), self.labels[idx], str(self.paths[idx])

In [5]:
def count_dataset_images(root, classes, val_folder_names=None, val_source="test_good"):
    val_folder_names = val_folder_names or ["validation", "val"]
    root_path = Path(root)
    stats = {}
    use_test_good_val = val_source == "test_good" and _has_test_good(root_path, classes)
    for cls in classes:
        cls_root = root_path / cls
        train_n = len(_list_images(cls_root / "train"))
        val_dir = _resolve_val_folder(cls_root, val_folder_names)
        if val_dir:
            val_n = len(_list_images(val_dir))
            val_src = "validation/"
        elif use_test_good_val:
            val_n = len(_list_images(cls_root / "test" / "good"))
            val_src = "test/good"
        else:
            val_n, val_src = 0, "—"
        test_root = cls_root / "test"
        test_n = len(_list_images(test_root)) if test_root.exists() else 0
        stats[cls] = {"train": train_n, "validation": val_n, "validation_source": val_src, "test": test_n}
    return stats

stats = count_dataset_images(
    str(DATASET_ROOT), list(cfg.data.classes),
    list(cfg.data.val_folder_names), cfg.data.get("val_source", "test_good"),
)
rows = "".join(
    f"<tr><td>{c}</td><td>{s['train']}</td><td>{s['validation']} ({s['validation_source']})</td><td>{s['test']}</td></tr>"
    for c, s in stats.items()
)
display(HTML("<table border=1><tr><th>Clase</th><th>train</th><th>val</th><th>test</th></tr>" + rows + "</table>"))
assert sum(s["train"] for s in stats.values()) > 0
assert sum(s["validation"] for s in stats.values()) > 0
print("Listo para entrenar.")

Clase,train,val,test
cable,224,58 (test/good),150
capsule,219,23 (test/good),132
screw,320,41 (test/good),160
transistor,213,60 (test/good),100


Listo para entrenar.


## 5 — W&B (credenciales de `.env.example`)

In [6]:
WANDB_PROJECT, WANDB_ENTITY = resolve_wandb_settings(
    cfg.logger.project, cfg.logger.entity, project_root=PROJECT_ROOT
)
api = wandb.Api()
_entity = WANDB_ENTITY or getattr(api, "default_entity", None)
WB_URL = f"https://wandb.ai/{_entity}/{WANDB_PROJECT}" if _entity else None
print("Workspace:", WB_URL)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\boyfa\_netrc
wandb: Currently logged in as: fabriciomena11 (fabriciomena11-tec) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


W&B OK — entity=fabriciomena11-tec, project=Tarea_3
Workspace: https://wandb.ai/fabriciomena11-tec/Tarea_3


## 6 — Modelos, visualización W&B y entrenamiento

In [7]:
from typing import Tuple

import torch
import torch.nn as nn
from pytorch_lightning import LightningModule, LightningDataModule
from torch.utils.data import DataLoader


class ReconstructionLoss(nn.Module):
    """Maneja diferentes tipos de pérdida: L1, L2, SSIM, SSIM+L1."""

    def __init__(self, loss_type: str = "l1", ssim_weight: float = 0.5, l1_weight: float = 0.5):
        super().__init__()
        self.loss_type = loss_type.lower()
        self.ssim_weight = ssim_weight
        self.l1_weight = l1_weight

        if self.loss_type == "l1":
            self.loss_fn = nn.L1Loss()
        elif self.loss_type == "l2":
            self.loss_fn = nn.MSELoss()
        elif self.loss_type == "ssim":
            self.loss_fn = None  # Usar SSIM directamente
        elif self.loss_type == "ssim_l1":
            self.loss_fn = nn.L1Loss()
        else:
            raise ValueError(f"Unknown loss type: {loss_type}")

    def forward(self, recon: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        if self.loss_type == "l1":
            return self.loss_fn(recon, target)
        elif self.loss_type == "l2":
            return self.loss_fn(recon, target)
        elif self.loss_type == "ssim":
            # SSIM: maximizar similitud = minimizar (1 - ssim)
            ssim_val = ssim(recon, target, data_range=1.0, size_average=True)
            return 1.0 - ssim_val
        elif self.loss_type == "ssim_l1":
            ssim_val = ssim(recon, target, data_range=1.0, size_average=True)
            l1_val = self.loss_fn(recon, target)
            return (1.0 - ssim_val) * self.ssim_weight + l1_val * self.l1_weight
        else:
            raise ValueError(f"Unknown loss type: {self.loss_type}")


class MVTecDataModule(LightningDataModule):
    """DataModule para MVTec AD con splits train/val/test."""

    def __init__(
        self,
        root: str,
        classes: list,
        image_size: int = 128,
        batch_size: int = 16,
        num_workers: int = 0,
        val_split: float = 0.1,
        use_predefined_splits: bool = True,
        val_folder_names: list = None,
        val_source: str = "test_good",
    ):
        super().__init__()
        self.root = root
        self.classes = classes
        self.image_size = image_size
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.val_split = val_split
        self.use_predefined_splits = use_predefined_splits
        self.val_folder_names = val_folder_names or ["validation", "val"]
        self.val_source = val_source

        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

    def setup(self, stage: str = None):
        """Crea los datasets para train, val y test."""
        if stage == "fit" or stage is None:
            self.train_dataset = MVTecImageDataset(
                root=self.root,
                classes=self.classes,
                image_size=self.image_size,
                split="train",
                val_folder_names=self.val_folder_names,
                val_source=self.val_source,
            )
            self.val_dataset = MVTecImageDataset(
                root=self.root,
                classes=self.classes,
                image_size=self.image_size,
                split="validation",
                val_folder_names=self.val_folder_names,
                val_source=self.val_source,
            )

        if stage == "test" or stage is None:
            self.test_dataset = MVTecImageDataset(
                root=self.root,
                classes=self.classes,
                image_size=self.image_size,
                split="test",
                val_folder_names=self.val_folder_names,
                val_source=self.val_source,
            )

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
        )

    def test_dataloader(self) -> DataLoader:
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
        )


In [8]:
from typing import Tuple

import torch
import torch.nn as nn
from pytorch_lightning import LightningModule


class ConvVAE(nn.Module):
    def __init__(self, in_channels: int = 3, latent_dim: int = 128):
        super().__init__()
        self.latent_dim = latent_dim
        self.enc = nn.Sequential(
            nn.Conv2d(in_channels, 32, 4, 2, 1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.ReLU(),
            nn.Conv2d(256, 512, 4, 2, 1),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(512 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(512 * 4 * 4, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, 512 * 4 * 4)
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, in_channels, 4, 2, 1),
            nn.Sigmoid(),
        )

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.enc(x).view(x.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        h = self.fc_dec(z).view(z.size(0), 512, 4, 4)
        return self.dec(h)

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar, z


class VAELightningModule(LightningModule):
    def __init__(
        self,
        latent_dim: int = 128,
        learning_rate: float = 1e-4,
        kl_weight: float = 0.001,
        loss_type: str = "l1",
        ssim_weight: float = 0.5,
        l1_weight: float = 0.5,
        in_channels: int = 3,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.model = ConvVAE(in_channels=in_channels, latent_dim=latent_dim)
        self.recon_loss = ReconstructionLoss(loss_type, ssim_weight, l1_weight)
        self.kl_weight = kl_weight
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch):
        x, _, _ = batch
        recon, mu, logvar, z = self.model(x)
        recon_l = self.recon_loss(recon, x)
        kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        loss = recon_l + self.kl_weight * kl
        return loss, recon, x, z

    def training_step(self, batch, batch_idx):
        loss, _, _, _ = self._shared_step(batch)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, recon, x, z = self._shared_step(batch)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return {"loss": loss, "recon": recon, "x": x, "z": z}

    def test_step(self, batch, batch_idx):
        loss, recon, x, z = self._shared_step(batch)
        self.log("test_loss", loss, on_step=False, on_epoch=True)
        return {"loss": loss, "recon": recon, "x": x, "z": z, "labels": batch[1]}

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

    def get_latent(self, x: torch.Tensor) -> torch.Tensor:
        mu, _ = self.model.encode(x)
        return mu


class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNetAutoencoder(nn.Module):
    """U-Net style autoencoder with skip connections for reconstruction."""

    def __init__(self, in_channels: int = 3, bottleneck_channels: int = 128):
        super().__init__()
        self.bottleneck_channels = bottleneck_channels
        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.down4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(512, bottleneck_channels)

        self.up4 = nn.ConvTranspose2d(bottleneck_channels, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, in_channels, 1)

    def forward(self, x: torch.Tensor):
        e1 = self.down1(x)
        e2 = self.down2(self.pool1(e1))
        e3 = self.down3(self.pool2(e2))
        e4 = self.down4(self.pool3(e3))
        b = self.bottleneck(self.pool4(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))
        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return torch.sigmoid(self.out(d1)), b.flatten(1)


class UNetAELightningModule(LightningModule):
    def __init__(
        self,
        bottleneck_channels: int = 128,
        learning_rate: float = 1e-4,
        loss_type: str = "l1",
        ssim_weight: float = 0.5,
        l1_weight: float = 0.5,
        in_channels: int = 3,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.model = UNetAutoencoder(in_channels, bottleneck_channels)
        self.recon_loss = ReconstructionLoss(loss_type, ssim_weight, l1_weight)
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch):
        x, _, _ = batch
        recon, latent = self.model(x)
        loss = self.recon_loss(recon, x)
        return loss, recon, x, latent

    def training_step(self, batch, batch_idx):
        loss, _, _, _ = self._shared_step(batch)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, recon, x, z = self._shared_step(batch)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return {"loss": loss, "recon": recon, "x": x, "z": z}

    def test_step(self, batch, batch_idx):
        loss, recon, x, z = self._shared_step(batch)
        self.log("test_loss", loss, on_step=False, on_epoch=True)
        return {"loss": loss, "recon": recon, "x": x, "z": z, "labels": batch[1]}

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

    def get_latent(self, x: torch.Tensor) -> torch.Tensor:
        _, z = self.model(x)
        return z


from typing import Any, Dict, List, Optional

import numpy as np
import wandb
from pytorch_lightning import Callback, Trainer
from pytorch_lightning.utilities import rank_zero_only
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


def _tensor_to_wandb_images(orig: torch.Tensor, recon: torch.Tensor, max_n: int = 16):
    n = min(max_n, orig.size(0))
    panels = []
    for i in range(n):
        o = (orig[i].detach().cpu().numpy().transpose(1, 2, 0) * 255).astype(np.uint8)
        r = (recon[i].detach().cpu().numpy().transpose(1, 2, 0) * 255).astype(np.uint8)
        panel = np.concatenate([o, r], axis=1)
        panels.append(wandb.Image(panel, caption=f"original|recon #{i}"))
    return panels


def _direct_log(key: str, value: Any) -> None:
    """Loggea directamente en el run activo de W&B sin pasar step (evita drops por step duplicado)."""
    if wandb.run is not None:
        wandb.run.log({key: value})


class WandbVisualizationCallback(Callback):
    """Logs reconstructions, t-SNE latent space, and test anomaly panels to W&B.

    FIXES aplicados vs versión original:
    1. Loss train+val: se acumulan TODAS las filas (train y val) y se loggea
       una única tabla al final de cada epoch de validación — ambas líneas visibles.
    2. Test images: se loggea sin step fijo (wandb.run.log sin step=) para evitar
       que W&B descarte silenciosamente el log por step ya usado tras trainer.fit().
    3. Defect reconstructions: el dataset de test usa label "good" (no "cable/good")
       para la subcarpeta good, así que la lógica is_good cubre ambos casos.
    """

    def __init__(
        self,
        num_images: int = 16,
        num_latent_samples: int = 200,
        log_tsne: bool = True,
        log_pca: bool = True,
    ):
        self.num_images = num_images
        self.num_latent_samples = num_latent_samples
        self.log_tsne = log_tsne
        self.log_pca = log_pca
        # Acumula TODAS las filas (train y val) para graficar 2 líneas en W&B
        self._all_loss_rows: List[List] = []

    # ──────────────────────────────────────────────────────────────
    # TRAIN
    # ──────────────────────────────────────────────────────────────
    def on_train_epoch_end(self, trainer: Trainer, pl_module: LightningModule) -> None:
        if trainer.sanity_checking:
            return
        train_loss = trainer.callback_metrics.get("train_loss_epoch") or trainer.callback_metrics.get("train_loss")
        if train_loss is not None:
            epoch = int(trainer.current_epoch)
            self._all_loss_rows.append([epoch, "train", float(train_loss)])
            # También loggear como scalar individual (visible en Charts de W&B)
            _direct_log("train/loss_epoch", float(train_loss))

    # ──────────────────────────────────────────────────────────────
    # VALIDATION
    # ──────────────────────────────────────────────────────────────
    def on_validation_epoch_end(self, trainer: Trainer, pl_module: LightningModule) -> None:
        if trainer.sanity_checking:
            return
        loader = trainer.datamodule.val_dataloader()
        pl_module.eval()
        device = pl_module.device

        # Recolectar TODO el dataloader (todas las clases)
        origs_all, recons_all, latents_all, labels_all = [], [], [], []
        with torch.no_grad():
            for batch in loader:
                x, labels, _ = batch
                x = x.to(device)
                if hasattr(pl_module, "model") and hasattr(pl_module.model, "encode"):
                    recon, mu, logvar, z = pl_module.model(x)
                    latent = mu
                else:
                    recon, latent = pl_module.model(x)
                origs_all.append(x.cpu())
                recons_all.append(recon.cpu())
                latents_all.append(latent.cpu())
                labels_all.extend(labels)

        if not origs_all:
            return

        origs_concat = torch.cat(origs_all)
        recons_concat = torch.cat(recons_all)
        latents_concat = torch.cat(latents_all)

        # Seleccionar 16 imágenes balanceadas por clase
        class_indices: Dict[str, List[int]] = {}
        for idx, lbl in enumerate(labels_all):
            cls = lbl.split("/")[0]
            class_indices.setdefault(cls, []).append(idx)

        selected_indices: List[int] = []
        imgs_per_class = max(1, self.num_images // max(len(class_indices), 1))
        for cls in sorted(class_indices):
            selected_indices.extend(class_indices[cls][:imgs_per_class])
        selected_indices = selected_indices[: self.num_images]
        idx_t = torch.tensor(selected_indices)

        orig_sel = origs_concat[idx_t]
        recon_sel = recons_concat[idx_t]
        latent_sample = latents_concat[: self.num_latent_samples]

        # 1) Reconstrucciones de validación (16 imágenes)
        panels = _tensor_to_wandb_images(orig_sel, recon_sel, self.num_images)
        _direct_log("val/reconstructions", panels)

        # 2) t-SNE y PCA del vector latente
        if latent_sample.size(0) >= 4:
            z_np = latent_sample.numpy()
            epoch = int(trainer.current_epoch)

            if self.log_pca and z_np.shape[1] >= 2:
                pca = PCA(n_components=2, random_state=42)
                emb_pca = pca.fit_transform(z_np)
                table_pca = wandb.Table(data=emb_pca.tolist(), columns=["pc1", "pc2"])
                _direct_log(
                    "val/latent_pca",
                    wandb.plot.scatter(
                        table_pca, "pc1", "pc2",
                        title=f"PCA latente epoch {epoch} (n={z_np.shape[0]})",
                    ),
                )

            if self.log_tsne:
                perplexity = min(30, max(2, z_np.shape[0] - 1))
                emb_tsne = TSNE(
                    n_components=2, perplexity=perplexity,
                    random_state=42, init="pca", learning_rate="auto",
                ).fit_transform(z_np)
                table_tsne = wandb.Table(data=emb_tsne.tolist(), columns=["x", "y"])
                _direct_log(
                    "val/latent_tsne",
                    wandb.plot.scatter(
                        table_tsne, "x", "y",
                        title=f"t-SNE latente epoch {epoch} (n={z_np.shape[0]})",
                    ),
                )

        # 3) Curva train+val loss — FIX: acumular filas de val y redibujar tabla completa
        val_loss = trainer.callback_metrics.get("val_loss")
        if val_loss is not None:
            epoch = int(trainer.current_epoch)
            val_scalar = float(val_loss)
            self._all_loss_rows.append([epoch, "val", val_scalar])
            # Scalar individual para Charts nativos de W&B
            _direct_log("val/loss_epoch", val_scalar)
            # Tabla acumulada con AMBAS líneas (train y val)
            if len(self._all_loss_rows) >= 2:
                table = wandb.Table(data=list(self._all_loss_rows), columns=["epoch", "split", "loss"])
                _direct_log(
                    "charts/train_val_loss",
                    wandb.plot.line(
                        table, "epoch", "loss", stroke="split",
                        title=f"Train vs Val Loss — epoch {epoch}",
                    ),
                )

    # ──────────────────────────────────────────────────────────────
    # TEST — FIX: no pasar step= para evitar drops silenciosos en W&B
    # ──────────────────────────────────────────────────────────────
    def on_test_epoch_end(self, trainer: Trainer, pl_module: LightningModule) -> None:
        loader = trainer.datamodule.test_dataloader()
        pl_module.eval()
        device = pl_module.device
        good_orig, good_recon = [], []
        defect_orig, defect_recon = [], []

        with torch.no_grad():
            for batch in loader:
                x, labels, _ = batch
                x = x.to(device)
                if hasattr(pl_module, "model") and hasattr(pl_module.model, "encode"):
                    recon, _, _, _ = pl_module.model(x)
                else:
                    recon, _ = pl_module.model(x)
                for i, lbl in enumerate(labels):
                    # FIX: el dataset usa label "good" (sin prefijo de clase) para test/good
                    # y también "cable/good" según la rama que ejecute; cubrimos ambos.
                    is_good = (lbl == "good") or lbl.endswith("/good")
                    t_o = x[i: i + 1].cpu()
                    t_r = recon[i: i + 1].cpu()
                    if is_good and len(good_orig) < self.num_images:
                        good_orig.append(t_o)
                        good_recon.append(t_r)
                    elif not is_good and len(defect_orig) < self.num_images:
                        defect_orig.append(t_o)
                        defect_recon.append(t_r)

        # FIX: usar _direct_log (sin step=) — W&B descarta logs con step ≤ último step de fit
        if good_orig:
            o = torch.cat(good_orig)[: self.num_images]
            r = torch.cat(good_recon)[: self.num_images]
            _direct_log("test/reconstructions_good", _tensor_to_wandb_images(o, r))

        if defect_orig:
            o = torch.cat(defect_orig)[: self.num_images]
            r = torch.cat(defect_recon)[: self.num_images]
            _direct_log("test/reconstructions_defect", _tensor_to_wandb_images(o, r))
        else:
            print(
                "[WandbVisualizationCallback] ADVERTENCIA: no se encontraron imágenes "
                "defectuosas en el set de test. Verifica que el dataset tenga subcarpetas "
                "de defectos en test/ distintas de 'good'."
            )

        # Loggear pérdida de test como scalar
        test_loss = trainer.callback_metrics.get("test_loss")
        if test_loss is not None:
            _direct_log("test/loss", float(test_loss))

        self._log_reconstruction_histograms(trainer, pl_module, loader, device)

    def _log_reconstruction_histograms(
        self, trainer: Trainer, pl_module: LightningModule, loader, device
    ) -> None:
        errors_good: List[float] = []
        errors_defect: Dict[str, List[float]] = {}

        with torch.no_grad():
            for batch in loader:
                x, labels, _ = batch
                x = x.to(device)
                if hasattr(pl_module, "model") and hasattr(pl_module.model, "encode"):
                    recon, _, _, _ = pl_module.model(x)
                else:
                    recon, _ = pl_module.model(x)
                err = torch.mean((recon - x) ** 2, dim=(1, 2, 3)).cpu().numpy()
                for e, lbl in zip(err, labels):
                    if (lbl == "good") or lbl.endswith("/good"):
                        errors_good.append(float(e))
                    else:
                        errors_defect.setdefault(lbl, []).append(float(e))

        hist_data = [["good", v] for v in errors_good]
        for defect_class, vals in errors_defect.items():
            hist_data.extend([[defect_class, v] for v in vals])

        if hist_data:
            table = wandb.Table(data=hist_data, columns=["class", "reconstruction_mse"])
            # FIX: sin step= para evitar drops en W&B
            _direct_log(
                "test/reconstruction_error_histogram",
                wandb.plot.histogram(
                    table, "reconstruction_mse",
                    title="Reconstruction error by class (test)",
                ),
            )


In [ ]:
"""Rutina de entrenamiento reutilizable desde Hydra CLI o Jupyter."""

from pathlib import Path
from typing import Optional

import pytorch_lightning as pl
import torch
from omegaconf import DictConfig, OmegaConf
from pytorch_lightning.loggers import WandbLogger



def build_model(cfg: DictConfig):
    loss_cfg = cfg.loss
    data_cfg = cfg.data
    if cfg.model.name == "vae":
        return VAELightningModule(
            latent_dim=cfg.model.latent_dim,
            learning_rate=cfg.model.learning_rate,
            kl_weight=cfg.model.kl_weight,
            loss_type=loss_cfg.type,
            ssim_weight=loss_cfg.get("ssim_weight", 0.5),
            l1_weight=loss_cfg.get("l1_weight", 0.5),
            in_channels=data_cfg.num_channels,
        )
    if cfg.model.name == "unet":
        return UNetAELightningModule(
            bottleneck_channels=cfg.model.bottleneck_channels,
            learning_rate=cfg.model.learning_rate,
            loss_type=loss_cfg.type,
            ssim_weight=loss_cfg.get("ssim_weight", 0.5),
            l1_weight=loss_cfg.get("l1_weight", 0.5),
            in_channels=data_cfg.num_channels,
        )
    raise ValueError(f"Modelo desconocido: {cfg.model.name}")


def build_datamodule(cfg: DictConfig) -> MVTecDataModule:
    return MVTecDataModule(
        root=cfg.data.root,
        classes=list(cfg.data.classes),
        image_size=cfg.data.image_size,
        batch_size=cfg.data.batch_size,
        num_workers=cfg.data.num_workers,
        val_split=cfg.data.val_split,
        use_predefined_splits=cfg.data.get("use_predefined_splits", True),
        val_folder_names=list(cfg.data.get("val_folder_names", ["validation", "val"])),
        val_source=cfg.data.get("val_source", "test_good"),
    )


def run_training(cfg: DictConfig, output_dir: Optional[Path] = None) -> str:
    """
    Entrena y evalúa con PyTorch Lightning + W&B.
    Devuelve el run id de W&B.
    """
    import os

    pl.seed_everything(cfg.seed, workers=True)
    from pathlib import Path as _Path

    root = _Path(cfg.data.root).parent if hasattr(cfg.data, "root") else _Path.cwd()
    wb_project, wb_entity = resolve_wandb_settings(
        project=os.environ.get("WANDB_PROJECT") or cfg.logger.project,
        entity=cfg.logger.entity,
        project_root=root,
    )

    if cfg.trainer.accelerator == "gpu" and not torch.cuda.is_available():
        raise RuntimeError(
            "Se configuró accelerator=gpu pero CUDA no está disponible. "
            "Instale PyTorch con soporte CUDA."
        )

    datamodule = build_datamodule(cfg)
    model = build_model(cfg)

    wandb_logger = WandbLogger(
        project=wb_project,
        entity=wb_entity,
        name=cfg.experiment_name,
        tags=list(cfg.logger.tags),
        save_dir=cfg.logger.save_dir,
        log_model=cfg.logger.log_model,
        config=OmegaConf.to_container(cfg, resolve=True),
    )

    trainer = pl.Trainer(
    max_epochs=cfg.trainer.max_epochs,
    accelerator=cfg.trainer.accelerator,
    devices=cfg.trainer.devices,
    precision=cfg.trainer.precision,
    gradient_clip_val=cfg.trainer.gradient_clip_val,
    log_every_n_steps=cfg.trainer.log_every_n_steps,
    check_val_every_n_epoch=cfg.trainer.check_val_every_n_epoch,
    enable_progress_bar=cfg.trainer.enable_progress_bar,
    logger=wandb_logger,
    callbacks=[WandbVisualizationCallback(
        num_images=cfg.trainer.num_images,
        num_latent_samples=cfg.trainer.num_latent_samples,
    )],
    )

    try:
        trainer.fit(model, datamodule=datamodule)
        trainer.test(model, datamodule=datamodule)
        run_id = wandb_logger.experiment.id
    finally:
        # Cierra el run activo para que el siguiente experimento cree uno nuevo.
        import wandb

        if wandb.run is not None:
            wandb.finish()

    if output_dir is not None:
        output_dir.mkdir(parents=True, exist_ok=True)
        (output_dir / "wandb_run_id.txt").write_text(run_id, encoding="utf-8")
    return run_id



def save_completed_runs(path: Path, completed: Dict[str, str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(completed, indent=2), encoding="utf-8")


def load_completed_runs(path: Path) -> Dict[str, str]:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))
"""Ejecuta la matriz de experimentos (Hydra) y persiste run ids."""

import json
from pathlib import Path
from typing import Dict, List, Optional

from omegaconf import OmegaConf

import os



def build_experiment_overrides(cfg) -> List[List[str]]:
    return [
        [f"model={m}", f"loss={l}"]
        for m in cfg.experiments.models
        for l in cfg.experiments.losses
    ]


def run_all_experiments(
    load_cfg_fn,
    project_root: Path,
    skip_if_complete: bool = True,
    force_retrain: bool = False,
) -> Dict[str, str]:
    """
    Entrena los 8 experimentos (2 modelos × 4 pérdidas).
    Devuelve {experiment_name: wandb_run_id}.
    """
    cfg_base = load_cfg_fn()
    resolve_wandb_settings(
        project=os.environ.get("WANDB_PROJECT") or cfg_base.logger.project,
        entity=cfg_base.logger.entity,
        project_root=project_root,
        force=force_retrain,
    )
    overrides_list = build_experiment_overrides(cfg_base)
    runs_file = project_root / "wandb_exports" / "completed_runs.json"
    completed = {} if force_retrain else load_completed_runs(runs_file)

    expected = len(overrides_list)
    if skip_if_complete and not force_retrain and len(completed) >= expected:
        print(f"Ya hay {len(completed)}/{expected} runs en {runs_file}.")
        print("Para re-entrenar: FORCE_RETRAIN = True en el notebook.")
        return completed

    pending = [ov for ov in overrides_list if load_cfg_fn(ov).experiment_name not in completed]
    print(f"Entrenamientos pendientes: {len(pending)} de {expected}")

    for ov in overrides_list:
        c = load_cfg_fn(ov)
        name = c.experiment_name
        if skip_if_complete and not force_retrain and name in completed:
            print(f"Omitiendo {name} (ya en completed_runs.json)")
            continue
        print("\n" + "=" * 60)
        print("Entrenando:", name)
        print(OmegaConf.to_yaml(c.trainer))
        run_id = run_training(c)
        completed[name] = run_id
        print("W&B run id:", run_id)
        save_completed_runs(runs_file, completed)

    return completed


## 7 — ENTRENAMIENTO (8 experimentos)

- `SKIP_TRAINING = True` → solo visualizar resultados previos
- `FORCE_RETRAIN = True` → repite los 8 aunque existan en `wandb_exports/completed_runs.json`

In [10]:
SKIP_TRAINING = False
FORCE_RETRAIN = True

if SKIP_TRAINING:
    COMPLETED_RUNS = load_completed_runs(PROJECT_ROOT / "wandb_exports" / "completed_runs.json")
    print("Entrenamiento omitido. Runs guardados:", len(COMPLETED_RUNS))
else:
    print("=" * 60)
    print("INICIANDO 8 ENTRENAMIENTOS (30 épocas c/u — varias horas)")
    print("=" * 60)
    COMPLETED_RUNS = run_all_experiments(
        load_cfg, PROJECT_ROOT,
        skip_if_complete=not FORCE_RETRAIN,
        force_retrain=FORCE_RETRAIN,
    )
    print("\n=== ENTRENAMIENTO COMPLETADO ===")
    for n, rid in COMPLETED_RUNS.items():
        print(f"  {n}: https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}/runs/{rid}")

INICIANDO 8 ENTRENAMIENTOS (30 épocas c/u — varias horas)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\boyfa\_netrc


W&B OK — entity=fabriciomena11-tec, project=Tarea_3


Seed set to 42


Entrenamientos pendientes: 8 de 8

Entrenando: vae_l1
max_epochs: 30
accelerator: gpu
devices: 1
precision: 16-mixed
gradient_clip_val: 1.0
log_every_n_steps: 10
check_val_every_n_epoch: 1
enable_progress_bar: true



...

Output()

c:\Users\boyfa\miniconda3\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\boyfa\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    0.2820565104484558     │
└───────────────────────────┴───────────────────────────┘

epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇█
test/loss,▁
test_loss,▁
train/loss_epoch,▇▃▁▃▁▁▃▂▂▂▂▂▁▂▂▁▁▁▁▅▄▅▄▅▆▅▇█▇▆
train_loss_epoch,▇▃▁▃▁▁▃▂▂▂▂▂▁▂▂▁▁▁▁▅▄▅▄▅▆▅▇█▇▆
train_loss_step,█▄▃▃▃▁▂▂▃▃▄▁▂▂▃▂▂▁▁▁▂▁▂▁▁▃▃▅▄▃▅▄▅▅▃▆▆▆▄▄
trainer/global_step,▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇██
val/loss_epoch,▂▂▂▂▁▁▂▂▂▂▂▂▂▂▂▂▂▁▂▃▃▄▄▅▄▃█▅▅▅
val_loss,▂▂▂▂▁▁▂▂▂▂▂▂▂▂▂▂▂▁▂▃▃▄▄▅▄▃█▅▅▅
epoch,30
test/loss,0.28206


W&B run id: rd38mq4l

=== ENTRENAMIENTO COMPLETADO ===
  vae_l1: https://wandb.ai/fabriciomena11-tec/Tarea_3/runs/y2fakzul
  vae_l2: https://wandb.ai/fabriciomena11-tec/Tarea_3/runs/w3tbh0fe
  vae_ssim: https://wandb.ai/fabriciomena11-tec/Tarea_3/runs/0ifx1y40
  vae_ssim_l1: https://wandb.ai/fabriciomena11-tec/Tarea_3/runs/3zj704qs
  unet_l1: https://wandb.ai/fabriciomena11-tec/Tarea_3/runs/t2ytdjm7
  unet_l2: https://wandb.ai/fabriciomena11-tec/Tarea_3/runs/r1u2w0kg
  unet_ssim: https://wandb.ai/fabriciomena11-tec/Tarea_3/runs/z4734hpv
  unet_ssim_l1: https://wandb.ai/fabriciomena11-tec/Tarea_3/runs/rd38mq4l


## 8 — Resultados y gráficas W&B

- Todas las gráficas e imágenes se **descargan de W&B** (sin matplotlib).
- Curvas train/val: panel `charts/train_val_loss` + Charts del run en wandb.ai.
- Use los enlaces para el dashboard completo en el navegador.

In [11]:
from IPython.display import display, HTML, Image
from pathlib import Path

# Diccionario con los títulos y los nombres de los archivos que debes descargar
archivos_graficas = {
    "Pérdida de Entrenamiento vs Validación": "train_val_loss.png",
    "Comparación de Reconstrucción (Validación - 16 imgs)": "val_reconstructions.png",
    "Comportamiento del Vector Latente (t-SNE)": "latent_tsne.png",
    "Reconstrucción Buenas (Set de Prueba - 16 imgs)": "test_recon_good.png",
    "Reconstrucción con Anomalías (Set de Prueba - 16 imgs)": "test_recon_defect.png",
    "Histogramas del Error de Reconstrucción": "error_histogram.png"
}

# Lista de los 8 experimentos
experimentos = [
    ("VAE", "L1", "vae_l1"),
    ("VAE", "L2", "vae_l2"),
    ("VAE", "SSIM", "vae_ssim"),
    ("VAE", "SSIM + L1", "vae_ssim_l1"),
    ("U-NET", "L1", "unet_l1"),
    ("U-NET", "L2", "unet_l2"),
    ("U-NET", "SSIM", "unet_ssim"),
    ("U-NET", "SSIM + L1", "unet_ssim_l1")
]

base_dir = Path("graficas_descargadas")

display(HTML("<h2>Evaluación de Resultados de los 8 Experimentos</h2>"))

for modelo, loss, carpeta in experimentos:
    # Genera un separador y el título del experimento
    display(HTML(f"<hr style='height:2px;border-width:0;color:gray;background-color:gray'>"))
    display(HTML(f"<h3 style='color:#2c3e50;'>Modelo: {modelo} | Función de Pérdida: {loss}</h3>"))
    
    exp_dir = base_dir / carpeta
    
    # Intenta cargar cada una de las 6 gráficas necesarias
    for titulo, archivo in archivos_graficas.items():
        img_path = exp_dir / archivo
        display(HTML(f"<h4>• {titulo}</h4>"))
        
        # Si la imagen existe, la muestra. Si no, crea un "espacio" indicando qué falta.
        if img_path.exists():
            display(Image(filename=str(img_path)))
        else:
            display(HTML(f"<div style='border:2px dashed #bdc3c7; padding:20px; margin-bottom:10px; text-align:center; color:#7f8c8d;'>"
                         f"<em>Espacio reservado para: <b>{archivo}</b></em><br>"
                         f"Guárdala en la ruta: {img_path}</div>"))

## 9 — Análisis crítico (completar)

Compare con las tablas e imágenes del paso 8:

- **L1 / L2 / SSIM / SSIM+L1** según `val_loss` y curvas train vs val
- **VAE vs U-Net** en reconstrucciones y proyecciones del latente (PCA y t-SNE)
- Separación **good vs defecto** en histogramas de error